# Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Current Registry

In [2]:
registry_df = pd.read_csv("current_rental_registrations_251001.csv")
registry_df["formatted_address"] = registry_df["RegisteredAddress"].str.replace(r"\s+,", ",", regex=True)
registry_df["formatted_address"] = registry_df["formatted_address"].astype(str).str.upper().str.strip()
# Remove unit numbers before the first comma (e.g., " AVE 5," → " AVE,")
registry_df["formatted_address"] = registry_df["formatted_address"].str.replace(
    r"\s+\d+(?=,)", "", regex=True
).str.strip()


In [3]:
import re

def clean_units(address):
    # Match pattern: [street] [unit], [city], MA [ZIP]
    match = re.match(r"^(.*\b(?:ST|AV|AVE|RD|BLVD|PL|CT|DR|TER|WAY|LN|SQ|TE|CIR|PKWY|PLZ|HWY))\s+[A-Z0-9\-]+, (.+?, MA \d{5})$", address)
    if match:
        return f"{match.group(1)}, {match.group(2)}"
    return address

registry_df["formatted_address"] = registry_df["formatted_address"].apply(clean_units)


In [4]:
registry_df['formatted_address'].sample(30)

3550                 60 QUEENSBERRY ST, BOSTON, MA 02215
39582              96 MORTON ST, JAMAICA PLAIN, MA 02130
29145             23 ASTICOU RD, JAMAICA PLAIN, MA 02130
32022              180 HAMILTON ST, DORCHESTER, MA 02122
20159                   2 CLARENDON ST, BOSTON, MA 02116
13671                108 BLUE HILL AV, ROXBURY, MA 02119
10015                7 VICTORIA ST, DORCHESTER, MA 02125
25917                    80 FENWOOD RD, BOSTON, MA 02115
18483                 40 CHAMPNEY ST, BRIGHTON, MA 02135
2696                30 PETERBOROUGH ST, BOSTON, MA 02215
33371                14 HERON ST, WEST ROXBURY, MA 02132
2592                     313 BEACON ST, BOSTON, MA 02116
9151               135 TOWNSEND ST, DORCHESTER, MA 02121
5915              414 SARATOGA ST, EAST BOSTON, MA 02128
2184                  26 PARK VALE AV, ALLSTON, MA 02134
26536             584 E THIRD ST, SOUTH BOSTON, MA 02127
11194    5078-5080 WASHINGTON ST, WEST ROXBURY, MA 02132
34221                118 HUNTIN

# Assessor Dataset

In [5]:
assessment_df = pd.read_csv("fy2025-property-assessment-data_12_30_2024.csv", dtype={21: str}, low_memory=False)

In [7]:
def combine_street_numbers(row):
    try:
        st_num = str(int(float(row['ST_NUM']))) if pd.notnull(row['ST_NUM']) else ""
        st_num2 = str(int(float(row['ST_NUM2']))) if pd.notnull(row['ST_NUM2']) else ""
        return f"{st_num}-{st_num2}" if st_num and st_num2 else st_num
    except:
        return ""

assessment_df["ST_NUM_COMBINED"] = assessment_df.apply(combine_street_numbers, axis=1)
assessment_df["ZIP_CODE_CLEAN"] = assessment_df["ZIP_CODE"].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

assessment_df["ST_NAME_CLEAN"] = assessment_df["ST_NAME"].astype(str).str.replace(r'\bAVE\.', 'AV', regex=True)

assessment_df["formatted_address"] = (
    assessment_df["ST_NUM_COMBINED"].str.strip() + " " +
    assessment_df["ST_NAME_CLEAN"].astype(str).str.strip() + ", " +
    assessment_df["CITY"].astype(str).str.strip() + ", MA " +
    assessment_df["ZIP_CODE_CLEAN"]
).str.upper()
assessment_df["formatted_address"].dropna().unique()[:10]

array(['104 PUTNAM ST, EAST BOSTON, MA 02128',
       '197 LEXINGTON ST, EAST BOSTON, MA 02128',
       '199 LEXINGTON ST, EAST BOSTON, MA 02128',
       '201 LEXINGTON ST, EAST BOSTON, MA 02128',
       '203 LEXINGTON ST, EAST BOSTON, MA 02128',
       '205-207 LEXINGTON ST, EAST BOSTON, MA 02128',
       '209-211 LEXINGTON ST, EAST BOSTON, MA 02128',
       '213 LEXINGTON ST, EAST BOSTON, MA 02128',
       '215 LEXINGTON ST, EAST BOSTON, MA 02128',
       '217 LEXINGTON ST, EAST BOSTON, MA 02128'], dtype=object)

In [8]:
assessment_df[["ST_NUM", "ST_NUM2", "ST_NAME", "CITY", "ZIP_CODE", "formatted_address"]].sample(50)


,ST_NUM,ST_NUM2,ST_NAME,CITY,ZIP_CODE,formatted_address
143210,11.0,NaN,EVERETT ST,JAMAICA PLAIN,2130.0,"11 EVERETT ST, JAMAICA PLAIN, MA 02130"
7889,65.0,NaN,LEWIS ST,EAST BOSTON,2128.0,"65 LEWIS ST, EAST BOSTON, MA 02128"
12609,6.0,NaN,ARMORY ST,CHARLESTOWN,2129.0,"6 ARMORY ST, CHARLESTOWN, MA 02129"
35304,146.0,NaN,Chandler ST,BOSTON,2116.0,"146 CHANDLER ST, BOSTON, MA 02116"
179911,58.0,NaN,HUNNEWELL AV,BRIGHTON,2135.0,"58 HUNNEWELL AV, BRIGHTON, MA 02135"
40737,118.0,NaN,RIVERWAY ST,BOSTON,2215.0,"118 RIVERWAY ST, BOSTON, MA 02215"
159783,43.0,NaN,CHILTON RD,WEST ROXBURY,2132.0,"43 CHILTON RD, WEST ROXBURY, MA 02132"
91709,3531.0,NaN,WASHINGTON ST,JAMAICA PLAIN,2130.0,"3531 WASHINGTON ST, JAMAICA PLAIN, MA 02130"
87025,242.0,NaN,S HUNTINGTON AV,JAMAICA PLAIN,2130.0,"242 S HUNTINGTON AV, JAMAICA PLAIN, MA 02130"
113425,28.0,NaN,SANTUIT ST,DORCHESTER,2124.0,"28 SANTUIT ST, DORCHESTER, MA 02124"


# Current vs Assessor MinHash

In [9]:
# STEP 0: Required Libraries
from datasketch import MinHash, MinHashLSH
import pandas as pd
import re

# STEP 1: Tokenizer + MinHash generator
def tokenize(address, ngram=3):
    address = re.sub(r'[^\w\s]', '', str(address).upper())
    return set(address[i:i+ngram] for i in range(len(address) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# STEP 2: Extract house number
def extract_house_number(address):
    match = re.match(r"^(\d+)", str(address).strip())
    return int(match.group(1)) if match else None

# STEP 3: Safe MinHash with house number check
def safe_minhash_match(address, lsh_index):
    try:
        tokens = tokenize(address)
        input_num = extract_house_number(address)
        m = create_minhash(tokens)
        matches = lsh_index.query(m)
        for match_addr in matches:
            match_num = extract_house_number(match_addr)
            if input_num == match_num:
                return match_addr
        return None
    except:
        return None



In [10]:
# Make sure your registry address column is cleaned and ready
registered_addresses = registry_df["formatted_address"].dropna().unique().tolist()

# Build LSH index
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in registered_addresses:
    m = create_minhash(tokenize(addr))
    lsh.insert(addr, m)



In [11]:
assessment_df["minhash_strict"] = assessment_df["formatted_address"].apply(lambda x: safe_minhash_match(x, lsh))

# Filter unmatched = unregistered
unregistered_assessor_strict = assessment_df[assessment_df["minhash_strict"].isnull()]

In [12]:
matched_assessor = assessment_df[assessment_df["minhash_strict"].notnull()]
matched_assessor[["formatted_address", "minhash_strict"]].head(20)


,formatted_address,minhash_strict
1,"197 LEXINGTON ST, EAST BOSTON, MA 02128","197 LEXINGTON ST, EAST BOSTON, MA 02128"
2,"199 LEXINGTON ST, EAST BOSTON, MA 02128","199 LEXINGTON ST, EAST BOSTON, MA 02128"
4,"203 LEXINGTON ST, EAST BOSTON, MA 02128","203 LEXINGTON ST, EAST BOSTON, MA 02128"
5,"205-207 LEXINGTON ST, EAST BOSTON, MA 02128","205-207 LEXINGTON ST, EAST BOSTON, MA 02128"
6,"209-211 LEXINGTON ST, EAST BOSTON, MA 02128","209 LEXINGTON ST, EAST BOSTON, MA 02128"
7,"213 LEXINGTON ST, EAST BOSTON, MA 02128","213 LEXINGTON ST, EAST BOSTON, MA 02128"
8,"215 LEXINGTON ST, EAST BOSTON, MA 02128","215 LEXINGTON ST, EAST BOSTON, MA 02128"
9,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"
10,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"
11,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"


In [13]:
total_assessor = assessment_df["formatted_address"].notna().sum()
unregistered_assessor_strict_count = len(unregistered_assessor_strict)
percent_unregistered_assessor = round((unregistered_assessor_strict_count / total_assessor) * 100, 2)
print("🏠 Assessor Properties")
print(f"Total formatted addresses: {total_assessor}")
print(f"Unregistered addresses: {unregistered_assessor_strict_count}")
print(f"Percentage unregistered: {percent_unregistered_assessor}%")


🏠 Assessor Properties
Total formatted addresses: 183442
Unregistered addresses: 127052
Percentage unregistered: 69.26%


# 311 Dataset

In [14]:
service_df = pd.read_csv("dff4d804-5031-443a-8409-8344efd0e5c8.csv", low_memory=False)

In [15]:
# Known city/neighborhood names to catch multi-word places like "SOUTH BOSTON", "JAMAICA PLAIN"
boston_neighborhoods = [
    "SOUTH BOSTON", "EAST BOSTON", "JAMAICA PLAIN", "MATTAPAN", "ROXBURY", 
    "BRIGHTON", "CHARLESTOWN", "HYDE PARK", "DORCHESTER", "WEST ROXBURY", 
    "ALLSTON", "ROSLINDALE", "BACK BAY", "FENWAY", "MISSION HILL", "NORTH END",
    "SOUTH END", "CHINATOWN"
]

def format_location(location):
    try:
        parts = location.strip().split()
        if len(parts) < 4:
            return location.upper()

        zip_code = parts[-1]
        state = parts[-2]

        # Try 2-word city names first
        possible_city = " ".join(parts[-4:-2]).upper()
        if possible_city in boston_neighborhoods:
            city = possible_city
            street = " ".join(parts[:-4])
        else:
            # Fall back to 1-word city names
            city = parts[-3].upper()
            street = " ".join(parts[:-3])

        return f"{street}, {city}, {state} {zip_code}".upper()
    except:
        return ""
service_df["formatted_address"] = service_df["location"].astype(str).apply(format_location)

# Replace AVE (or AVE.) with AV in 311 formatted addresses
service_df["formatted_address"] = service_df["formatted_address"].str.replace(
    r"\bAVE\.?\b", "AV", regex=True
)


service_df["formatted_address"].sample(30)

75288                976R-976 RIVER ST, HYDE PARK, MA 02136
174032             711 E SEVENTH ST, SOUTH BOSTON, MA 02127
240040    INTERSECTION OF PIERPONT RD & THEODORE PARKER ...
158632    INTERSECTION OF DARTMOUTH ST & BOYLSTON, ST, B...
278250                   292R FOSTER ST, BRIGHTON, MA 02135
54239               25 CHESTNUT HILL AV, BRIGHTON, MA 02135
125631               18 NATIONAL ST, SOUTH BOSTON, MA 02127
134061                108 SOUTHERN AV, DORCHESTER, MA 02124
263626                      91 LORNA RD, MATTAPAN, MA 02126
167766               2093 CENTRE ST, WEST ROXBURY, MA 02132
210330    INTERSECTION OF PUBLIC ALLEY NO. 441 & PUBLIC ...
178576                        7 EXETER ST, BOSTON, MA 02116
174944                    249 FOSTER ST, BRIGHTON, MA 02135
235671               52-54 HARVEST ST, DORCHESTER, MA 02125
69585                    25 SALUTATION ST, BOSTON, MA 02109
48511           3440 WASHINGTON ST, JAMAICA PLAIN, MA 02130
243996                      7 CONCORD SQ

# Current Vs 311 MinHash

In [16]:
# STEP 0: Required Libraries
from datasketch import MinHash, MinHashLSH
import pandas as pd
import re

# STEP 1: Tokenizer + MinHash generator
def tokenize(address, ngram=3):
    address = re.sub(r'[^\w\s]', '', str(address).upper())
    return set(address[i:i+ngram] for i in range(len(address) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# STEP 2: Extract house number
def extract_house_number(address):
    match = re.match(r"^(\d+)", str(address).strip())
    return int(match.group(1)) if match else None

# STEP 3: Safe MinHash with house number check
def safe_minhash_match(address, lsh_index):
    try:
        tokens = tokenize(address)
        input_num = extract_house_number(address)
        m = create_minhash(tokens)
        matches = lsh_index.query(m)
        for match_addr in matches:
            match_num = extract_house_number(match_addr)
            if input_num == match_num:
                return match_addr
        return None
    except:
        return None


In [17]:
# Make sure your registry address column is cleaned and ready
registered_addresses = registry_df["formatted_address"].dropna().unique().tolist()

# Build LSH index
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in registered_addresses:
    m = create_minhash(tokenize(addr))
    lsh.insert(addr, m)


In [18]:
# Apply strict MinHash match with house number check
service_df["minhash_strict"] = service_df["formatted_address"].apply(lambda x: safe_minhash_match(x, lsh))

# Filter unmatched = unregistered
unregistered_311_strict = service_df[service_df["minhash_strict"].isnull()]

matched_311 = service_df[service_df["minhash_strict"].notnull()]

# Show side-by-side comparison
matched_311[["formatted_address", "minhash_strict"]].head(20)


,formatted_address,minhash_strict
2,"416 BELGRADE AV, WEST ROXBURY, MA 02132","416 BELGRADE AV, WEST ROXBURY, MA 02132"
3,"3 SAINT CHARLES ST, BOSTON, MA 02116","3 SAINT CHARLES ST, BOSTON, MA 02116"
9,"4 HARTWELL ST, DORCHESTER, MA 02121","4-6 HARTWELL ST, DORCHESTER, MA 02121"
10,"32 FAIRFIELD ST, BOSTON, MA 02116","32 FAIRFIELD ST, BOSTON, MA 02116"
17,"193 W NINTH ST, SOUTH BOSTON, MA 02127","193 W EIGHTH ST, SOUTH BOSTON, MA 02127"
19,"40 MORRIS ST, EAST BOSTON, MA 02128","40 MORRIS ST, EAST BOSTON, MA 02128"
20,"4 GILMER ST, MATTAPAN, MA 02126","4 GILMER ST, MATTAPAN, MA 02126"
25,"394 MERIDIAN ST, EAST BOSTON, MA 02128","394 MERIDIAN ST, EAST BOSTON, MA 02128"
27,"1200 WASHINGTON ST, ROXBURY, MA 02118","1200 WASHINGTON ST, ROXBURY, MA 02118"
29,"331 FANEUIL ST, BRIGHTON, MA 02135","331 FANEUIL ST, BRIGHTON, MA 02135"


In [19]:
total_311 = service_df["formatted_address"].notna().sum()
unregistered_311_strict_count = len(unregistered_311_strict)
percent_unregistered_311 = round((unregistered_311_strict_count / total_311) * 100, 2)
print("📋 311 Service Requests")
print(f"Total formatted addresses: {total_311}")
print(f"Unregistered addresses: {unregistered_311_strict_count}")
print(f"Percentage unregistered: {percent_unregistered_311}%\n")

📋 311 Service Requests
Total formatted addresses: 282836
Unregistered addresses: 220260
Percentage unregistered: 77.88%



# Unregistered List Concat

In [20]:
# Get formatted addresses from both unregistered datasets
unreg_311_addresses = unregistered_311_strict["formatted_address"].dropna()
unreg_assessor_addresses = unregistered_assessor_strict["formatted_address"].dropna()

# Concatenate and drop duplicates
combined_unregistered = pd.concat([unreg_311_addresses, unreg_assessor_addresses]).drop_duplicates().sort_values()

# Print result
print(f"✅ Total unique unregistered addresses (combined): {len(combined_unregistered)}")
print("\n📌 Sample addresses:")
print(combined_unregistered.head(10).to_string(index=False))


✅ Total unique unregistered addresses (combined): 105588

📌 Sample addresses:
                                    
              A ST, BOSTON, MA 02210
        A ST, SOUTH BOSTON, MA 02127
         ABBY RD, BRIGHTON, MA 02135
 ACADEMY HILL RD, BRIGHTON, MA 02135
   ACADIA ST, SOUTH BOSTON, MA 02127
          ACORN ST, BOSTON, MA 02108
       ACTON ST, HYDE PARK, MA 02136
        ADA ST, ROSLINDALE, MA 02131
          ADAMS PL, BOSTON, MA 02114


In [21]:
combined_unregistered.to_csv("combined_unregistered_addresses.csv", index=False)


# SAM_ID Match

In [22]:
import pandas as pd
from datasketch import MinHash, MinHashLSH
import re

# Load your datasets
unregistered_df = pd.read_csv("combined_unregistered_addresses.csv")
sam_df = pd.read_csv("live_street_address_management_sam_addresses.csv")

# Extract just house number + street name
def extract_street_only(address):
    try:
        return str(address).split(",")[0].strip().upper()
    except:
        return ""

unregistered_df["street_only"] = unregistered_df.iloc[:, 0].apply(extract_street_only)
sam_df["street_only"] = sam_df["FULL_ADDRESS"].astype(str).apply(extract_street_only)

# Tokenization and MinHash functions
def tokenize(text, ngram=3):
    text = re.sub(r'[^\w\s]', '', str(text).upper())
    return set(text[i:i+ngram] for i in range(len(text) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# Build LSH index from SAM street-only addresses
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in sam_df["street_only"].dropna().unique():
    lsh.insert(addr, create_minhash(tokenize(addr)))

# Match each unregistered street address
def match_street_only(text):
    try:
        m = create_minhash(tokenize(text))
        results = lsh.query(m)
        return results[0] if results else None
    except:
        return None

unregistered_df["minhash_match"] = unregistered_df["street_only"].apply(match_street_only)

# Merge matched SAM ID back in
matched_df = unregistered_df.merge(
    sam_df[["street_only", "SAM_ADDRESS_ID", "FULL_ADDRESS"]],
    left_on="minhash_match",
    right_on="street_only",
    how="left"
)

# Total unregistered addresses
total_unregistered = len(unregistered_df)

# Matched rows (those that got a SAM ID)
matched_with_sam = matched_df["SAM_ADDRESS_ID"].notna().sum()

# Calculate percentage
percent_matched = round((matched_with_sam / total_unregistered) * 100, 2)

# Print results
print("📊 SAM ID Matching Summary")
print(f"- Total unregistered addresses: {total_unregistered}")
print(f"- Matched with SAM ID: {matched_with_sam}")
print(f"- Percentage matched: {percent_matched}%")



/var/folders/4h/gm1b6b155z73jqkthlx4bgv00000gn/T/ipykernel_74698/3324297159.py:7: DtypeWarning: Columns (6,7,15,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  sam_df = pd.read_csv("live_street_address_management_sam_addresses.csv")


📊 SAM ID Matching Summary
- Total unregistered addresses: 105587
- Matched with SAM ID: 91147
- Percentage matched: 86.32%


In [23]:
matched_df.to_csv("unregistered_with_sam_ids.csv", index=False)
print("✅ Done! Output saved to 'unregistered_with_sam_ids.csv'")

✅ Done! Output saved to 'unregistered_with_sam_ids.csv'
